In [5]:
pip install --user pysam

  Using cached pysam-0.23.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (1.6 kB)
Using cached pysam-0.23.0-cp311-cp311-manylinux_2_28_x86_64.whl (26.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pysam
import csv

# This script makes variant_sequences.tsv that has the variants specified in variant_table_file and a 17 bp sequence in which the reference (from reference_fasta) and alternate alleles sits in the middle of
# Paths to your files
reference_fasta = "/home/newol210/macknight_oliver/braker_working_dir/UTM_Trep_v1.0_hap1_genomic.fa"
vcf_file = "/home/newol210/macknight_oliver/NoneVsMidVsHigh_RNASeq/Processed_Fastq_by_fastp_attempt2/fastp_processedReads_mapped_to_UTM_hap1/all_bams/sorted_and_indexed_bams/variants_LowVsHigh_genomeWide/variants_renamedSamples_sorted.vcf.gz"
variant_table_file = "/home/newol210/macknight_oliver/NoneVsMidVsHigh_RNASeq/Processed_Fastq_by_fastp_attempt2/fastp_processedReads_mapped_to_UTM_hap1/all_bams/sorted_and_indexed_bams/variants_LowVsHigh_genomeWide/cosegregating_variants_CM061887_chromosome_POS.csv"

# Load reference genome
fasta = pysam.FastaFile(reference_fasta)

# Load VCF
vcf = pysam.VariantFile(vcf_file)

# Read variant table into list of (chrom, pos)
variant_list = []
with open(variant_table_file, 'r') as f:
    reader = csv.reader(f, delimiter='\t')
    for row in reader:
        chrom = row[0]
        pos = int(row[1])
        variant_list.append((chrom, pos))

# Open output file
with open('variant_sequences.tsv', 'w', newline='') as out_f:
    writer = csv.writer(out_f, delimiter='\t')
    writer.writerow(['Chromosome', 'Position', 'Ref_sequence', 'Alt_sequence'])

    for chrom, pos in variant_list:
        # Look up variant in VCF
        rec_found = False
        for rec in vcf.fetch(chrom, pos - 1, pos):
            if rec.pos == pos:
                rec_found = True
                ref_allele = rec.ref
                alt_allele = rec.alts[0]  # just first ALT allele
                break

        if not rec_found:
            print(f"Variant at {chrom}:{pos} not found in VCF — skipping")
            continue

        len_ref = len(ref_allele)

        # Get 8bp upstream
        flank_left = fasta.fetch(chrom, pos - 9, pos - 1)

        # Get 8bp downstream (after end of REF allele)
        flank_right = fasta.fetch(chrom, pos - 1 + len_ref, pos - 1 + len_ref + 8)

        # Build sequences
        ref_seq = flank_left + ref_allele + flank_right
        alt_seq = flank_left + alt_allele + flank_right

        writer.writerow([chrom, pos, ref_seq.upper(), alt_seq.upper()])


In [12]:
import csv

# This script outputs just alterate alleles in fasta format (caps_markers.fasta)
# Path to the output table from the CAPS script
input_table = '/home/newol210/macknight_oliver/NoneVsMidVsHigh_RNASeq/Processed_Fastq_by_fastp_attempt2/fastp_processedReads_mapped_to_UTM_hap1/all_bams/sorted_and_indexed_bams/variants_LowVsHigh_genomeWide/variant_sequences.tsv'  # <-- change if your table has a different name

# Path to the FASTA file you want to create
output_fasta = 'caps_markers.fasta'

# Open the table and write the FASTA
with open(input_table, 'r') as infile, open(output_fasta, 'w') as fasta_out:
    reader = csv.DictReader(infile, delimiter='\t')
    for row in reader:
        position = row['Position']
        alt_seq = row['Alt_sequence']
        fasta_out.write(f'>{position}\n{alt_seq}\n')

print(f'Done! FASTA file written to {output_fasta}')


Done! FASTA file written to caps_markers.fasta


In [18]:
import csv

# This script outputs both reference and alterate alleles in fasta format (caps_markers_ref_alt.fasta)
# Input CAPS marker table
input_table = '/home/newol210/macknight_oliver/NoneVsMidVsHigh_RNASeq/Processed_Fastq_by_fastp_attempt2/fastp_processedReads_mapped_to_UTM_hap1/all_bams/sorted_and_indexed_bams/variants_LowVsHigh_genomeWide/cosegregating_variants_CM061887.1_chromosome_POS_sequences.tsv'  # <-- or your current output table name

# Output FASTA
output_fasta = 'caps_markers_ref_and_alt.fasta'

with open(input_table, 'r') as infile, open(output_fasta, 'w') as fasta_out:
    reader = csv.DictReader(infile, delimiter='\t')
    for row in reader:
        position = row['Position']
        ref_seq = row['Ref_sequence']
        alt_seq = row['Alt_sequence']
        
        # Write ref sequence
        fasta_out.write(f'>{position}_ref\n{ref_seq}\n')
        # Write alt sequence
        fasta_out.write(f'>{position}_alt\n{alt_seq}\n')

print(f'Done! FASTA file with ref and alt sequences written to {output_fasta}')


Done! FASTA file with ref and alt sequences written to caps_markers_ref_and_alt.fasta


In [16]:
from collections import defaultdict

# This script marks variants that are in a 8 bp proximity to other variants
# This cell only works with the fasta made from just the alternate alleles!
# Input FASTA (from previous step)
input_fasta = '/home/newol210/macknight_oliver/NoneVsMidVsHigh_RNASeq/Processed_Fastq_by_fastp_attempt2/fastp_processedReads_mapped_to_UTM_hap1/all_bams/sorted_and_indexed_bams/variants_LowVsHigh_genomeWide/cosegregating_variants_chr5s_chromosome_POS_sequences_ref_alt.fasta'
output_fasta = '/home/newol210/macknight_oliver/NoneVsMidVsHigh_RNASeq/Processed_Fastq_by_fastp_attempt2/fastp_processedReads_mapped_to_UTM_hap1/all_bams/sorted_and_indexed_bams/variants_LowVsHigh_genomeWide/cosegregating_variants_chr5s_chromosome_POS_sequences_ref_alt_proximity_marked.fasta'

# Function to read FASTA into a dict: {position: sequence}
def read_fasta(fasta_file):
    seqs = {}
    with open(fasta_file, 'r') as f:
        header = None
        seq_lines = []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if header is not None:
                    seqs[header] = ''.join(seq_lines)
                header = line[1:]  # Remove '>'
                seq_lines = []
            else:
                seq_lines.append(line)
        if header is not None:
            seqs[header] = ''.join(seq_lines)
    return seqs

# Step 1: Read FASTA into dict
sequences = read_fasta(input_fasta)

# Step 2: Extract and sort positions (as integers)
positions = sorted([int(pos) for pos in sequences.keys()])

# Step 3: Identify positions that are within 8 bp of any other position
proximity_marked = set()
for i, pos1 in enumerate(positions):
    for j in range(i+1, len(positions)):
        pos2 = positions[j]
        if abs(pos2 - pos1) <= 8:
            proximity_marked.add(pos1)
            proximity_marked.add(pos2)
        elif pos2 - pos1 > 8:
            break  # No need to keep going (since sorted)

# Step 4: Write new FASTA with asterisks
with open(output_fasta, 'w') as out:
    for pos_str, seq in sequences.items():
        pos = int(pos_str)
        if pos in proximity_marked:
            header = f'>{pos}*'
        else:
            header = f'>{pos}'
        out.write(f'{header}\n{seq}\n')

print(f'Done! Marked FASTA written to {output_fasta}')


ValueError: invalid literal for int() with base 10: '578127_ref'